In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd

In [2]:
def build_t2_set(L1, L2, R1, c1, R2, c2, P, mu, h1):
    m_star = math.floor((mu-h1*L1)*R2/c2)
    sub_list = []
    for m in range(0, m_star+1):
        sub_list.append(m)

    return sub_list



In [3]:
def build_decision_set(L1, L2, R1, c1, R2, c2, P, mu):
    dec_list = []
    h = 0
    while h <= 1:
        sub = build_t2_set(L1, L2, R1, c1, R2, c2, P, mu, h)
        for j in sub:
            dec_list.append((h, j))

        h += 0.01
    return dec_list
        

In [4]:
def utility(L1, L2, R1, c1, R2, c2, P, mu, h1, m, i_max=None, eps=1e-12):

    #TYPE 2
    n_star = math.floor((mu-h1*L1)*R2/c2)
    lam1 = L1 * h1
    if not (lam1 < mu):
        raise ValueError("Stability requires L1*h1 < mu.")
        

    dim = n_star + 1
    I = np.eye(dim)
    
    A1 = lam1 * I
    Am1 = mu * I
    A0 = -(lam1 + mu) * I 

    R = np.zeros((dim, dim))
    A0_inv = np.linalg.inv(A0)
    for _ in range(2000):
        R_next = -(A1 + (R @ R) @ Am1) @ A0_inv
        if np.linalg.norm(R_next - R, ord=np.inf) < 1e-14:
            R = R_next
            break
        R = R_next

    R_list = [None] * (n_star + 1)
    R_list[n_star] = R
    for i in range(n_star - 1, 0, -1):
        A0_n = np.zeros((dim, dim))
        for j in range(dim):
            rate2 = L2 if j < (n_star - i) else 0
            A0_n[j, j] = -(lam1 + rate2 + mu)
            if j < dim - 1:
                A0_n[j, j+1] = rate2
        R_list[i] = -A1 @ np.linalg.inv(A0_n + R_list[i+1] @ Am1)

    sigma = -(lam1 + L2)
    beta = np.zeros((1, dim)); beta[0, 0] = (lam1 + L2)
    b = np.zeros((dim, 1)); b[0, 0] = mu
    B0 = np.zeros((dim, dim))
    for j in range(dim):
        rate2 = L2 if j < n_star else 0
        B0[j, j] = -(lam1 + rate2 + mu)
        if j < n_star: B0[j, j+1] = rate2
        if j > 0: B0[j, j-1] = mu
    B0[0,0] = -(lam1 + L2 + mu)

    if n_star >= 1:
        K = B0 + R_list[1] @ Am1 - (1/sigma) * (b @ beta)
    else:
        K = B0 + R @ Am1 - (1/sigma) * (b @ beta)
    vals, vecs = np.linalg.eig(K.T)
    pi_0 = np.real(vecs[:, np.argmin(np.abs(vals))])
    pi_0 = pi_0 / np.sum(pi_0)

    pi_levels = [None] * (n_star + 1)
    pi_levels[0] = pi_0
    for i in range(1, n_star + 1):
        pi_levels[i] = pi_levels[i-1] @ R_list[i]

    pi_empty_val = -float(np.dot(pi_0, b).item()) / sigma
    
    total = pi_empty_val + sum(np.sum(p) for p in pi_levels[:n_star])
    total += np.sum(pi_levels[n_star] @ np.linalg.inv(I - R))
    
    pi_empty_final = pi_empty_val / total
    pi_final = [p / total for p in pi_levels]

    p_join2 = pi_empty_final
    pi000 = float(pi_empty_final)
    results = {(0,0,0):pi000}

    for i in range(n_star + 1):
        for j in range(dim):
            prob = pi_final[i][j]
            if i+j+1 <= n_star:
                p_join2 += prob
                results[(i,j,1)] = float(prob)
            

    h2 = float(p_join2)
    #TYPE 1
    U1 =  R1 - c1*((L1*h1 + L2*h2)/(mu*(mu-L1*h1)))
    W1 = (L1*h1 + L2*h2)/(mu*(mu-L1*h1))

    state = list(results.keys())
    U2 = 0
    W2 = 0
    for s in state:
        i, j, k = s[0], s[1], s[2]
        if i + j + k <= n_star:
            W2 += results[s]*(  (i+j+k)/(mu - L1*h1)    )
            U2 += results[s]*( R2 - c2*( (i+j+k)/(mu - L1*h1)  )  )

    return h1, h2, U1, U2, W1, W2
    

In [5]:
def max_util(L1, L2, R1, c1, R2, c2, P, mu):
    decision_set = build_decision_set(L1, L2, R1, c1, R2, c2, P, mu)
    results = {}

    best_value = float("-inf")
    best_decision = None
    
    for dec in decision_set:
        hh, mm = dec
        h1, h2, U1, U2, W1, W2 = utility(L1, L2, R1, c1, R2, c2, P, mu, hh, mm)
        U = L1*h1*U1 + L2*h2*U2
        results[dec] = U

        if U > best_value:
            best_value = U
            best_decision = dec
            best_h = h1, h2
            best_W = W1, W2

    return best_h, best_decision, best_value, best_W



        

In [6]:
max_util(1, 1, 5, 7, 10, 3, 0, 3)

((0.5900000000000003, 0.9980667859694131),
 (0.5900000000000003, 0),
 10.642864115887752,
 (0.21964962461540993, 0.45466430574966565))